
**How to use the widget:**

1. Replace `"path_to_your_model"` with the actual path to your fine-tuned Whisper model.
2. Run the code in a Jupyter Notebook.
3. Press and hold the spacebar to record; a red waveform will indicate recording.
4. Release the spacebar to stop recording and transcribe the audio using your fine-tuned model.
5. The transcription appears in the output area below the visualization.

**Notes:**
- The visualization uses a simulated waveform (red during recording, blue when idle) with a dark background for clarity on both light and dark themes.
- The widget uses your model’s pipeline configuration, matching your example code.
- Ensure `pyaudio`, `librosa`, `transformers`, `torch`, and `ipywidgets` are installed.
- The temporary audio file (`temp_recording.wav`) is saved in the working directory and overwritten with each recording.
- If you encounter issues with the model loading, verify the model path and dependencies.



In [5]:
#in colab
#!apt-get install -y portaudio19-dev
#!pip install pyaudio

In [6]:
import pyaudio
import wave
import librosa
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline

In [7]:
# Audio recording settings
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 16000
WAVE_OUTPUT_FILENAME = "temp_recording.wav"
MODEL_PATH = "model/"  # Update with your model path

In [8]:

# Load Whisper model and processor
print("🔹 Loading fine-tuned model...")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH)
processor = WhisperProcessor.from_pretrained(MODEL_PATH)
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0 if torch.cuda.is_available() else -1,
    generate_kwargs={"forced_decoder_ids": None}
)
print("✅ Fine-tuned model loaded successfully")


🔹 Loading fine-tuned model...


Device set to use cpu


✅ Fine-tuned model loaded successfully


In [ ]:

import time

duration = int(input("Enter recording duration in seconds: "))
audio = pyaudio.PyAudio()
stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)
frames = []
print(f"Recording for {duration} seconds...")

for _ in range(0, int(RATE / CHUNK * duration)):
    data = stream.read(CHUNK, exception_on_overflow=False)
    frames.append(data)

print("Recording stopped.")
stream.stop_stream()
stream.close()
audio.terminate()

# Save the recorded audio
wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
wf.setnchannels(CHANNELS)
wf.setsampwidth(audio.get_sample_size(FORMAT))
wf.setframerate(RATE)
wf.writeframes(b''.join(frames))
wf.close()

# Transcribe audio
print("Transcribing...")
audio_data, sr = librosa.load(WAVE_OUTPUT_FILENAME, sr=16000)
result = pipe(audio_data)
transcription = result['text'].strip()
print("Transcription:", transcription)